# Finished Main Class Voice Test

This notebook verifies that the local voice helper module can be imported and used to generate a WAV file from a text prompt.

In [1]:
import gc
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
from pathlib import Path
import importlib
import sys
import librosa
import numpy as np
import torch
from torch.utils.data import Dataset

# TTS imports - Coqui TTS adaptation path
from TTS.api import TTS





/home/tmichel3796/anaconda3/envs/spark_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tmichel3796/anaconda3/envs/spark_env/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
import os
os.getcwd()
directory = '/mnt/d/my_repos/college/business_programming/big_project'
#directory ='/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project'  
#os.chdir('/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project')
os.chdir(directory)
os.getcwd()
from finish_use_voice import create_voice_file, VoiceFileService
from finished_pi_sensor import RaspberryPiSensor


## test voice file.

In [3]:
voice_service = VoiceFileService(project_root=directory)
voice_file_path = voice_service.create_voice_file(
    text="The light sensor reads steady while the temperature stays comfortable for the room.",
    output_filename="sensor_reading.wav"
)

print(f"Voice file created: {voice_file_path}")
voice_file_path

Loading weights: 100%|█████████████████████████████████████████████████████████████| 396/396 [00:00<00:00, 67727.30it/s]
SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Voice file created: /mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/synthesized_audio/sensor_reading.wav


PosixPath('/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/synthesized_audio/sensor_reading.wav')

## Example Raspberry Pi workflow

This example shows how the finished notebook would read the Pi sensors and have the Pi speak a message if a Raspberry Pi were available.

In [3]:
# Example values only. Replace these with your actual Raspberry Pi IP and password.
PI_HOST = "192.168.0.16"
PI_USERNAME = "pi"
PI_PASSWORD = "Solar3clipse!"

# Connect to the Pi
pi = RaspberryPiSensor(
    host=PI_HOST,
    username=PI_USERNAME,
    password=PI_PASSWORD,
    prompt_for_password=False,
)

# Read the sensors
pi.get_light_reading(light_sensor_port=0),
pi.get_temperature_reading(dht_sensor_port=5, dht_sensor_type=0)


# Build a message for the Pi to speak
message = (
    f"The light level is {pi.get_light_reading(light_sensor_port=0).get('value')} and "
    f"the temperature is {pi.get_temperature_reading(dht_sensor_port=5, dht_sensor_type=0).get('temperature_c'):.1f} degrees Celsius."
)

# Create a voice file locally, then copy or place it on the Pi if needed
voice_service = VoiceFileService(project_root=directory)
voice_file_path = voice_service.create_voice_file(
    text=message,
    output_filename="pi_status_message.wav",
)

print(f"Voice file created: {voice_file_path}")

# If the WAV file is available on the Pi itself, you can play it there.
# This example assumes the audio file has already been copied to the Pi.
# pi.play_audio_file("/home/pi/pi_status_message.wav", blocking=True)

pi.disconnect()

RuntimeError: Pi command failed with exit status 1. STDERR: Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import json,time,grovepi;value=int(grovepi.analogRead(0));print(json.dumps({'sensor':'light','port':0,'value':value,'timestamp':time.time()}))
    ^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'grovepi'